## Пререквизиты

In [1]:
import logging
import os
import json
import hashlib
import re
from pathlib import Path
import datetime
import random
import gc

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import pandas as pd 
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report, confusion_matrix
import emoji

In [2]:
logging.basicConfig(level=logging.INFO,force=True)

logger = logging.getLogger(__name__)


In [3]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
with open('/kaggle/input/datasets/timurx/train-dataset/training_dataset.json', 'r', encoding='utf-8') as f:

    dataset = json.load(f)

texts = [item["text"] for item in dataset]
labels = [item["label"] for item in dataset]

In [5]:
with open('/kaggle/input/datasets/timurx/test-dataset/golden_dataset_classifier.json', 'r', encoding='utf-8') as f:

    golden_dataset = json.load(f)

golden_texts = [item["text"] for item in golden_dataset]
golden_labels = [item["label"] for item in golden_dataset]

In [6]:
with open('/kaggle/input/datasets/timurx/aug-dataset/augmented_dataset.jsonl', 'r', encoding='utf-8') as f:
    aug_dataset = [json.loads(line) for line in f]

aug_texts = [item['text'] for item in aug_dataset]
aug_labels = [item['label'] for item in aug_dataset]

In [7]:
class RuBertClassifier:
    """Классификатор на основе RuBERT"""

    def __init__(self, repo_id: str = None):
        self.classes_ = ["single", "listing", "junk"]
        self.label2id = {l: i for i, l in enumerate(self.classes_)}
        self.id2label = {i: l for l, i in self.label2id.items()}

        if repo_id:
            self.tokenizer = AutoTokenizer.from_pretrained(
                repo_id,
            )
            self.model = AutoModelForSequenceClassification.from_pretrained(
                repo_id,
                num_labels=3,           
            )
        else:
            # загружаем из репозитория DeepPavlov по умолчанию
            self.tokenizer = AutoTokenizer.from_pretrained(
                "DeepPavlov/rubert-base-cased-conversational"
            )
            self.model = AutoModelForSequenceClassification.from_pretrained(
                "DeepPavlov/rubert-base-cased-conversational",
                num_labels=3,
                hidden_dropout_prob=0.4,            
                attention_probs_dropout_prob=0.4,
                attn_implementation="eager",
            )

        self.processor = TextProcessor()
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = self.model.to(self.device)
        logger.info(f"RuBERT загружен на {self.device}\n")

    def train(self, texts, labels, num_epochs=10, batch_size=8, lr=2e-5, patience=2):
        """Train с early stopping (по Macro F1) и сохранением лучшей модели"""
        if not texts or not labels:
            return {"eval_macro_f1": 0.0}

        logger.info(f"Датасет: {len(texts)} samples")

        # предобработка текста
        texts = [self.processor.preprocess(t) for t in texts]
        label_ids = torch.tensor([self.label2id[l] for l in labels])
        y_integers = [self.label2id[l] for l in labels]

        class_weights = compute_class_weight(
            class_weight="balanced", classes=np.unique(y_integers), y=y_integers
        )
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(
            self.device
        )
        logger.info(f"Calculated class weights: {self.class_weights}")

        # токенизация
        logger.info("Токенизация")
        encodings = self.tokenizer(
            texts, truncation=True, max_length=512, padding=True, return_tensors="pt"
        )

        input_ids = encodings["input_ids"]
        attention_mask = encodings["attention_mask"]

        (
            train_inputs,
            val_inputs,
            train_labels,
            val_labels,
            train_masks,
            val_masks,
        ) = train_test_split(
            input_ids,
            label_ids,
            attention_mask,
            test_size=0.2,
            random_state=42,
            stratify=label_ids,
        )

        train_dataset = TensorDataset(
            train_inputs.to(self.device),
            train_masks.to(self.device),
            train_labels.to(self.device),
        )
        val_dataset = TensorDataset(
            val_inputs.to(self.device),
            val_masks.to(self.device),
            val_labels.to(self.device),
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)

        logger.info(f"Train: {len(train_dataset)}, Val (Early Stop): {len(val_dataset)}\n")

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=0.01)

        total_steps = len(train_loader) * num_epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(total_steps * 0.1),
            num_training_steps=total_steps,
        )

        logger.info("Обучение\n")
        best_f1 = 0
        best_model_state = None
        no_improve = 0
        learning_curve = []
        for epoch in range(num_epochs):
            # train
            self.model.train()
            train_loss = 0

            for input_id, mask, label in tqdm(
                train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"
            ):
                optimizer.zero_grad()
                outputs = self.model(input_ids=input_id, attention_mask=mask)
                loss = F.cross_entropy(outputs.logits, label, weight=self.class_weights)
                loss.backward()
                optimizer.step()
                scheduler.step()
                train_loss += loss.item()

            # evaluation
            self.model.eval()
            val_loss = 0
            all_preds = []
            all_labels = []

            with torch.no_grad():
                for input_id, mask, label in val_loader:
                    outputs = self.model(input_ids=input_id, attention_mask=mask)
                    logits = outputs.logits
                    loss = F.cross_entropy(logits, label, weight=self.class_weights)
                    val_loss += loss.item()

                    preds = torch.argmax(logits, dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(label.cpu().numpy())

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)

            report_dict = classification_report(
                all_labels, all_preds, target_names=self.classes_, output_dict=True, zero_division=0
            )
            macro_f1 = report_dict['macro avg']['f1-score']

            logger.info(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {macro_f1:.4f}")
            learning_curve.append({
                "epoch": epoch + 1,
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "val_macro_f1": macro_f1,
                "learning_rate": scheduler.get_last_lr()[0] 
            })
            
            if macro_f1 > best_f1:
                best_f1 = macro_f1
                best_model_state = self.model.state_dict()
                no_improve = 0
                logger.info(f"Лучшая модель сохранена (Macro F1 = {best_f1:.4f})\n")
            else:
                no_improve += 1
                logger.info(f"нет улучшений: {no_improve}/{patience})\n")

            if no_improve >= patience:
                logger.info(f"Ранняя остановка на эпохе {epoch + 1}")
                break

        if best_model_state:
            self.model.load_state_dict(best_model_state)

        logger.info("Обучение завершено")
        return learning_curve

    def predict(self, text, return_probs=False):
        """Предсказание класса текста"""
        if not text:
            return "junk", 0.0

        text = self.processor.preprocess(text)

        inputs = self.tokenizer(
            text, return_tensors="pt", truncation=True, max_length=512, padding=True
        ).to(self.device)

        with torch.no_grad():
            outputs = self.model(**inputs)

        logits = outputs.logits[0]
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        pred_id = np.argmax(probs)

        if return_probs:
            return self.id2label[pred_id], {
                self.id2label[i]: float(probs[i]) for i in range(3)
            }
        return self.id2label[pred_id], float(probs[pred_id])

    def save(self, path):
        self.model.save_pretrained(f"{path}/model")
        self.tokenizer.save_pretrained(f"{path}/tokenizer")
        logger.info(f"Модель сохранена в {path}")

class TextProcessor:
    """Предобработка текста для RuBERT"""

    def __init__(self):
        self.html_pattern = re.compile(r"<[^>]+>")

        self.video_url_pattern = re.compile(
            r"https?://(?:www\.)?(?:youtube\.com|youtu\.be|vk\.com/video|rutube\.ru|vimeo\.com|music\.yandex\.ru|podcasts\.apple\.com)[^\s]+"
        )
        self.url_pattern = re.compile(r"https?://[^\s]+")

        self.mail_pattern = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")
        self.phone_pattern = re.compile(r"[\+]?[0-9\s\-\(\)]{10,}")
        self.multiple_spaces = re.compile(r"\s+")

    def preprocess(self, text: str) -> str:
        """Предобработка текста для классификации (Transformer-friendly)"""
        if not isinstance(text, str) or len(text) == 0:
            return "empty"

        

        text = text.lower()
        text = emoji.demojize(text, language="ru")
        
        text = self.html_pattern.sub(" ", text)
        text = self.video_url_pattern.sub(" [MEDIA_URL] ", text)
        text = self.url_pattern.sub(" [URL] ", text)
        text = self.mail_pattern.sub(" [EMAIL] ", text)
        text = self.phone_pattern.sub(" [PHONE] ", text)

        text = self.multiple_spaces.sub(" ", text)

        text = text.strip()

        if len(text) <= 10:
            return "empty"

        words = text.split()

        if len(words) <= 400:
            return text

        return " ".join(words[:400])


## Обучение

In [8]:
def plant_seed(seed: int = 42):
    """Фиксируем все генераторы случайных чисел."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [9]:
def train_and_test(train_texts, train_labels, test_texts, test_labels, seeds, 
                   aug_texts=None, aug_labels=None,
                   tapt=True, aug_addition=False):
    
    has_aug_data = bool(aug_texts and aug_labels)
    actual_aug_addition = aug_addition and has_aug_data

    experiment_results = {
        "metadata": {
            "timestamp": str(datetime.datetime.now()),
            "tapt_enabled": tapt,
            "aug_enabled": actual_aug_addition,
            "seeds_used": seeds,
            "train_size": len(train_texts) + (len(aug_texts) if actual_aug_addition else 0),
            "test_size": len(test_texts),
            "aug_size": len(aug_texts) if has_aug_data else 0
        },
        "runs": [] 
    }

    if tapt:
        repo_id = "/kaggle/input/models/timurx/rubert-tapt-pretrained/pytorch/default/1"
    else:
        repo_id = None

    if actual_aug_addition:
        current_train_texts = train_texts + aug_texts
        current_train_labels = train_labels + aug_labels
    else:
        current_train_texts = train_texts 
        current_train_labels = train_labels
        
    global_best_f1 = 0.0 
        
    for seed in seeds:
        logger.info(f"Запуск эксперимента с seed={seed}")
        plant_seed(seed) 
        
        clf = RuBertClassifier(repo_id)
        learning_curve = clf.train(current_train_texts, current_train_labels, num_epochs=10, patience=2)

        logger.info(f"Оценка на {len(test_texts)} текстах золотого датасета")
        preds = []
        for text in tqdm(test_texts, desc=f"Инференс"):
            pred_class, _ = clf.predict(text) 
            preds.append(pred_class)

        report_dict = classification_report(test_labels, preds, target_names=clf.classes_, output_dict=True)
        cm = confusion_matrix(test_labels, preds, labels=clf.classes_)
        
        current_macro_f1 = report_dict['macro avg']['f1-score']
        logger.info(f"Macro F1 на золотом датасете для seed={seed}: {current_macro_f1:.4f}")

        if current_macro_f1 > global_best_f1:
            global_best_f1 = current_macro_f1
            logger.info(f"Исторически лучший результат (Macro F1: {global_best_f1:.4f}). Сохраняем модель")
            best_model_dir = f"/kaggle/working/best_model_tapt_{tapt}_aug_{actual_aug_addition}"
            clf.save(best_model_dir)

        run_data = {
            "seed": seed,
            "golden_classification_report": report_dict,
            "golden_confusion_matrix": cm.tolist(),
            "learning_curve": learning_curve
        }
        
        if not actual_aug_addition and has_aug_data:            
            sample_size = min(1000, len(aug_texts)) 
            sampled_indices = random.sample(range(len(aug_texts)), sample_size)

            logger.info(f"Оценка на {sample_size} текстах синтетического датасета")
            random_aug_text = [aug_texts[i] for i in sampled_indices]
            random_aug_labels = [aug_labels[i] for i in sampled_indices]
            
            aug_preds = []
            for text in tqdm(random_aug_text, desc=f"Инференс"):
                pred_class, _ = clf.predict(text)
                aug_preds.append(pred_class)
                
            aug_report_dict = classification_report(random_aug_labels, aug_preds, target_names=clf.classes_, output_dict=True)
            aug_cm = confusion_matrix(random_aug_labels, aug_preds, labels=clf.classes_)
            
            aug_macro_f1 = aug_report_dict['macro avg']['f1-score']
            logger.info(f"Macro F1 на синтетическом датасете для seed={seed}: {aug_macro_f1:.4f}")
            
            run_data["aug_classification_report"] = aug_report_dict
            run_data["aug_confusion_matrix"] = aug_cm.tolist()

        experiment_results["runs"].append(run_data)

        del clf
        gc.collect()
        torch.cuda.empty_cache()

    output_file = f"/kaggle/working/experiment_metrics_tapt_{tapt}_aug_{actual_aug_addition}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(experiment_results, f, ensure_ascii=False, indent=4)
            
    return experiment_results

In [10]:
seeds = [i for i in range(40, 45)]

In [11]:
# чтобы не обучать все за одну сессию и не упереться в лимиты
RUN_BASE_MODEL = False
RUN_TAPT_MODEL = False
RUN_AUG_MODEL  = False  
RUN_TAPT_AUG_MODEL =True

In [12]:
if RUN_TAPT_MODEL:
    experiment_results = train_and_test(train_texts=texts, train_labels=labels, test_texts=golden_texts, 
                                     test_labels=golden_labels, aug_texts=aug_texts, aug_labels=aug_labels, 
                                     seeds=seeds, tapt=True, aug_addition=False)

In [13]:
if RUN_BASE_MODEL:
    experiment_results = train_and_test(train_texts=texts, train_labels=labels, test_texts=golden_texts, 
                                     test_labels=golden_labels, aug_texts=aug_texts, aug_labels=aug_labels, 
                                     seeds=seeds, tapt=False, aug_addition=False)

In [14]:
if RUN_AUG_MODEL:
    experiment_results = train_and_test(train_texts=texts, train_labels=labels, test_texts=golden_texts, 
                                     test_labels=golden_labels, aug_texts=aug_texts, aug_labels=aug_labels, 
                                     seeds=seeds, tapt=False, aug_addition=True)

In [15]:
if RUN_TAPT_AUG_MODEL:
    experiment_results = train_and_test(train_texts=texts, train_labels=labels, test_texts=golden_texts, 
                                     test_labels=golden_labels, aug_texts=aug_texts, aug_labels=aug_labels, 
                                     seeds=seeds, tapt=True, aug_addition=True)

INFO:__main__:Запуск эксперимента с seed=40


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/models/timurx/rubert-tapt-pretrained/pytorch/default/1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your dow

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:__main__:Модель сохранена в /kaggle/working/best_model_tapt_True_aug_True
INFO:__main__:Запуск эксперимента с seed=41


...

BertForSequenceClassification LOAD REPORT from: /kaggle/input/models/timurx/rubert-tapt-pretrained/pytorch/default/1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your dow

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:__main__:Модель сохранена в /kaggle/working/best_model_tapt_True_aug_True
INFO:__main__:Запуск эксперимента с seed=44


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/models/timurx/rubert-tapt-pretrained/pytorch/default/1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your dow

In [16]:
#!kaggle datasets delete timurx/aug-dataset -y